# Gas Station Project Finance Model

This notebook reproduces a simple investment model for a new gas station in Saint Petersburg.

**Market references**
- Bank of Russia key rate: **15.00%** on 2026-04-23
- Saint Petersburg AI-95 retail range: **68.06–71.70 RUB/l** on 2026-04-20
- Model price used: midpoint = **69.88 RUB/l**

Everything else is shown explicitly as a model assumption.


In [ ]:
import pandas as pd
import numpy as np
import numpy_financial as npf
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
KEY_RATE = 0.15
DISCOUNT_RATE = 0.18
CAPEX = 18_000_000
DAILY_VOLUME = 3000
RETAIL_PRICE = 69.88
MARGIN_PER_LITER = 8.0
ANNUAL_OPEX = 6_000_000
YEARS = 10

annual_liters = DAILY_VOLUME * 365
annual_revenue = annual_liters * RETAIL_PRICE
annual_gross_profit = annual_liters * MARGIN_PER_LITER
annual_ebitda = annual_gross_profit - ANNUAL_OPEX
cash_flows = [-CAPEX] + [annual_ebitda] * YEARS
discounted = [cf / ((1 + DISCOUNT_RATE) ** i) for i, cf in enumerate(cash_flows)]

projection = pd.DataFrame({
    "year": list(range(0, YEARS + 1)),
    "revenue_rub": [0] + [annual_revenue] * YEARS,
    "gross_profit_rub": [0] + [annual_gross_profit] * YEARS,
    "opex_rub": [0] + [ANNUAL_OPEX] * YEARS,
    "cash_flow_rub": cash_flows,
    "discounted_cash_flow_rub": discounted,
})
projection["cumulative_cash_flow_rub"] = projection["cash_flow_rub"].cumsum()
projection["cumulative_discounted_cash_flow_rub"] = projection["discounted_cash_flow_rub"].cumsum()

projection


In [ ]:
metrics = {
    "Annual liters": annual_liters,
    "Annual revenue, RUB": annual_revenue,
    "Annual gross profit, RUB": annual_gross_profit,
    "Annual EBITDA, RUB": annual_ebitda,
    "NPV, RUB": npf.npv(DISCOUNT_RATE, cash_flows),
    "IRR": npf.irr(cash_flows),
    "Simple payback, years": CAPEX / annual_ebitda,
}
metrics


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(projection["year"], projection["cumulative_cash_flow_rub"] / 1e6, marker="o", label="Cumulative cash flow")
plt.plot(projection["year"], projection["cumulative_discounted_cash_flow_rub"] / 1e6, marker="o", label="Cumulative discounted cash flow")
plt.axhline(0, linewidth=1)
plt.title("Cumulative project cash flow")
plt.xlabel("Year")
plt.ylabel("RUB million")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
volume_rows = []
for volume in range(1500, 5001, 250):
    annual_ebitda_v = volume * 365 * MARGIN_PER_LITER - ANNUAL_OPEX
    cash_flows_v = [-CAPEX] + [annual_ebitda_v] * YEARS
    volume_rows.append({
        "daily_volume_liters": volume,
        "npv_rub": npf.npv(DISCOUNT_RATE, cash_flows_v),
        "irr": npf.irr(cash_flows_v),
    })
volume_df = pd.DataFrame(volume_rows)

plt.figure(figsize=(10, 5))
plt.plot(volume_df["daily_volume_liters"], volume_df["npv_rub"] / 1e6, marker="o")
plt.axhline(0, linewidth=1)
plt.title("NPV sensitivity to daily throughput")
plt.xlabel("Daily volume, liters")
plt.ylabel("NPV, RUB million")
plt.tight_layout()
plt.show()

volume_df.head()


In [ ]:
margin_rows = []
for margin in np.arange(5, 11.5, 0.5):
    annual_ebitda_m = DAILY_VOLUME * 365 * margin - ANNUAL_OPEX
    cash_flows_m = [-CAPEX] + [annual_ebitda_m] * YEARS
    margin_rows.append({
        "margin_rub_per_liter": margin,
        "npv_rub": npf.npv(DISCOUNT_RATE, cash_flows_m),
        "irr": npf.irr(cash_flows_m),
    })
margin_df = pd.DataFrame(margin_rows)

plt.figure(figsize=(10, 5))
plt.plot(margin_df["margin_rub_per_liter"], margin_df["npv_rub"] / 1e6, marker="o")
plt.axhline(0, linewidth=1)
plt.title("NPV sensitivity to gross margin per liter")
plt.xlabel("Margin, RUB per liter")
plt.ylabel("NPV, RUB million")
plt.tight_layout()
plt.show()

margin_df.head()


## Conclusion

With the current assumptions, the base case does **not** pass the investment hurdle rate:
- NPV is negative
- IRR is below the discount rate

So the model points to the need for a higher daily throughput, a higher margin per liter, or a lower cost base.
